# Perceptron From Scratch

## What is a Perceptron?

A perceptron is the simplest possible classifier. It draws a straight line through your data
and says: everything on one side is class 0, everything on the other side is class 1.

Think of it like a bouncer at a club:
> "If your score is above this threshold, you're in (class 1). Otherwise, you're out (class 0)."

It is the building block of neural networks. A neural network is just many perceptrons stacked together.

## How does it work?

A perceptron takes input features, multiplies each by a weight, adds a bias, and checks the result:

```
z      = w1*x1 + w2*x2 + ... + wn*xn + b    (weighted sum)
y_hat  = 1  if z >= 0
         0  if z <  0                         (step function)
```

## How does it learn?

It looks at each training example, makes a prediction, and if it is wrong it nudges the weights:

```
error  = y_true - y_hat          (0 if correct, +1 or -1 if wrong)
w      = w + lr * error * x      (push weights in the right direction)
b      = b + lr * error          (push bias too)
```

If the prediction was correct, error = 0 and nothing changes.
If it was wrong, the weights shift just enough to try to fix it.

## What we build:
| Step | What happens |
|------|--------------|
| 1 | Create a linearly separable dataset |
| 2 | Implement the perceptron update rule from scratch |
| 3 | Train and watch the weights change |
| 4 | Plot the decision boundary |
| 5 | Compare with sklearn |
| 6 | Show where the perceptron fails (XOR) |

## Step 1 — The Dataset

We create a simple dataset where two classes are clearly separated by a straight line.
This is called a **linearly separable** dataset — the perceptron can only solve these.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)

# Class 0: points around (1, 1)
class0 = np.random.randn(20, 2) + np.array([1, 1])
# Class 1: points around (4, 4)
class1 = np.random.randn(20, 2) + np.array([4, 4])

X = np.vstack([class0, class1])          # shape (40, 2)
y = np.array([0]*20 + [1]*20)            # labels

plt.figure(figsize=(6, 5))
plt.scatter(class0[:, 0], class0[:, 1], label='Class 0', marker='o')
plt.scatter(class1[:, 0], class1[:, 1], label='Class 1', marker='s')
plt.title('Our Dataset (linearly separable)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 2 — The Perceptron: Weights and Activation

We start with weights and bias all set to zero.

The **step function** is the activation:
- If the weighted sum >= 0 → predict class 1
- If the weighted sum < 0  → predict class 0

This is the simplest possible decision rule.

In [ ]:
def step_function(z):
    return 1 if z >= 0 else 0

def predict_one(x, w, b):
    z = np.dot(w, x) + b
    return step_function(z)

def predict(X, w, b):
    return np.array([predict_one(x, w, b) for x in X])

# Test with zero weights — everything predicts 1 (z=0 >= 0)
w = np.zeros(2)
b = 0.0
print('Weights:', w, '  Bias:', b)
print('Sample predictions before training:', predict(X[:5], w, b))
print('True labels:                        ', y[:5])

## Step 3 — The Update Rule

For each training example:
1. Make a prediction
2. Compute the error: `error = y_true - y_hat`
   - If correct: error = 0, nothing changes
   - If predicted 0 but true is 1: error = +1, push weights up
   - If predicted 1 but true is 0: error = -1, push weights down
3. Update: `w = w + lr * error * x` and `b = b + lr * error`

We repeat this for several **epochs** (full passes through the data).
The perceptron is **guaranteed to converge** if the data is linearly separable.

In [ ]:
def train_perceptron(X, y, lr=0.1, epochs=20):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []   # track accuracy each epoch

    for epoch in range(1, epochs + 1):
        errors = 0
        for xi, yi in zip(X, y):
            y_hat = predict_one(xi, w, b)
            error = yi - y_hat
            if error != 0:
                w = w + lr * error * xi
                b = b + lr * error
                errors += 1

        acc = np.mean(predict(X, w, b) == y)
        history.append(acc)

        if epoch <= 5 or acc == 1.0:
            print(f'Epoch {epoch:2d}  |  Misclassified: {errors:2d}  |  Accuracy: {acc:.2f}  |  w={w.round(3)}  b={round(b,3)}')
            if acc == 1.0:
                print(f'          Converged at epoch {epoch}!')
                break

    return w, b, history

w, b, history = train_perceptron(X, y, lr=0.1, epochs=50)

## Step 4 — Accuracy Over Epochs

We can see how quickly the perceptron learns.
Once it reaches 100% accuracy on training data, it has found a separating line and stops updating.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(history) + 1), history, marker='o')
plt.title('Perceptron Training Accuracy Per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1.05)
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 5 — The Decision Boundary

The decision boundary is the line where `z = 0`:

```
w1*x1 + w2*x2 + b = 0
=> x2 = (-w1*x1 - b) / w2
```

Everything above this line is predicted as one class, everything below as the other.
The perceptron learned this line purely from the data.

In [ ]:
x1_vals = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)

# Decision boundary: w1*x1 + w2*x2 + b = 0  =>  x2 = (-w1*x1 - b) / w2
x2_boundary = (-w[0] * x1_vals - b) / w[1]

plt.figure(figsize=(6, 5))
plt.scatter(class0[:, 0], class0[:, 1], label='Class 0', marker='o')
plt.scatter(class1[:, 0], class1[:, 1], label='Class 1', marker='s')
plt.plot(x1_vals, x2_boundary, 'r--', linewidth=2, label='Decision boundary')
plt.title('Learned Decision Boundary')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

final_acc = np.mean(predict(X, w, b) == y)
print(f'Final accuracy: {final_acc * 100:.1f}%')
print(f'Learned weights: w = {w.round(4)},  b = {round(b, 4)}')

## Step 6 — Compare With sklearn

sklearn's `Perceptron` uses the same algorithm.
The weights may look different (different random seed, different order of updates)
but the decision boundary should separate the data just as well.

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score

sk_model = Perceptron(max_iter=50, eta0=0.1, random_state=42)
sk_model.fit(X, y)

sk_acc = accuracy_score(y, sk_model.predict(X))
print(f'sklearn Perceptron accuracy: {sk_acc * 100:.1f}%')
print(f'sklearn weights: {sk_model.coef_[0].round(4)}')
print(f'sklearn bias:    {sk_model.intercept_[0].round(4)}')
print()
print(f'Our weights:     {w.round(4)}')
print(f'Our bias:        {round(b, 4)}')

## Step 7 — Where the Perceptron Fails: The XOR Problem

The perceptron can **only** learn a straight line boundary.
If the data cannot be separated by a straight line, it will never converge.

The classic example is **XOR** (exclusive or):

| x1 | x2 | Output |
|----|----|---------|
| 0  | 0  | 0 |
| 0  | 1  | 1 |
| 1  | 0  | 1 |
| 1  | 1  | 0 |

Class 1 points are at (0,1) and (1,0) — diagonally opposite corners.
You cannot draw a single straight line to separate them from (0,0) and (1,1).

This limitation is why we need **multi-layer neural networks** — multiple perceptrons
stacked together can learn non-linear boundaries.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

# Try to train a perceptron on XOR
print('Training perceptron on XOR...')
w_xor = np.zeros(2)
b_xor = 0.0
lr = 0.1

for epoch in range(1, 21):
    errors = 0
    for xi, yi in zip(X_xor, y_xor):
        y_hat = predict_one(xi, w_xor, b_xor)
        error = yi - y_hat
        if error != 0:
            w_xor = w_xor + lr * error * xi
            b_xor = b_xor + lr * error
            errors += 1
    acc = np.mean(predict(X_xor, w_xor, b_xor) == y_xor)
    print(f'  Epoch {epoch:2d}  |  Misclassified: {errors}  |  Accuracy: {acc:.2f}')

print()
print('The perceptron keeps making mistakes — it cannot solve XOR.')
print('The accuracy never reaches 1.0 because no straight line can separate XOR.')

In [ ]:
# Visualize why XOR is impossible for a perceptron
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: our linearly separable dataset (perceptron works)
axes[0].scatter(class0[:, 0], class0[:, 1], label='Class 0', marker='o', s=60)
axes[0].scatter(class1[:, 0], class1[:, 1], label='Class 1', marker='s', s=60)
x1_vals = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
axes[0].plot(x1_vals, (-w[0] * x1_vals - b) / w[1], 'r--', linewidth=2, label='Decision boundary')
axes[0].set_title('Linear data — Perceptron WORKS')
axes[0].legend()
axes[0].grid(True)

# Right: XOR (perceptron fails)
colors = ['blue' if label == 0 else 'orange' for label in y_xor]
markers = ['o' if label == 0 else 's' for label in y_xor]
for xi, yi, c, m in zip(X_xor, y_xor, colors, markers):
    axes[1].scatter(xi[0], xi[1], color=c, marker=m, s=200, zorder=5)
    axes[1].annotate(f'({int(xi[0])},{int(xi[1])})=>{yi}', xi,
                     textcoords='offset points', xytext=(8, 5), fontsize=11)
axes[1].set_xlim(-0.5, 1.5)
axes[1].set_ylim(-0.5, 1.5)
axes[1].set_title('XOR data — Perceptron FAILS\n(no straight line can separate this)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Summary

| Concept | Explanation |
|---------|-------------|
| **Weighted sum** | z = w1*x1 + w2*x2 + b |
| **Step function** | Output 1 if z >= 0, else 0 |
| **Error** | y_true - y_hat (0, +1, or -1) |
| **Weight update** | w = w + lr * error * x |
| **Bias update** | b = b + lr * error |
| **Convergence** | Guaranteed only if data is linearly separable |
| **XOR failure** | Shows why we need multi-layer networks |

**Key insight:**
The perceptron is the ancestor of all neural networks.
Its update rule is a simplified version of gradient descent.
Its failure on XOR was the original motivation for inventing hidden layers —
which is exactly what `Neural_Networks_Simple_Intuitive_Introduction.ipynb` covers next.